# 03. Model Training & Evaluation

## Overview
This notebook serves as the main entry point for:
1.  **Verifying the Hardware** (GPU availability).
2.  **Training the ResNet50 Model or efficientnet_b3** (training pipeline).
3.  **Evaluating Performance** using professional diagnostic plots (Confusion Matrix, ROC, Reliability Curves).

## Objectives
-   Confirm CUDA acceleration is active.
-   Train the model on the processed dataset.
-   Assess generalization performance on the validation set.
-   Visualize class-wise metrics and prediction confidence.

In [ ]:
# --- Imports ---
%load_ext autoreload
%autoreload 2

import sys
import os
import shutil
from pathlib import Path
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

# Add project root to path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.config import (
    PROJECT_ROOT, MODELS_DIR, REPORTS_DIR,
    QUALITY_THRESHOLD, PROCESSED_DATA_DIR, DATA_DIR,
)
from src.utils.logger import get_logger
from src.visualization.plots import (
    set_plot_style,
    plot_learning_curves,
    plot_model_diagnostics,
    plot_confidence_abstention_panel,
    plot_binary_classification_metrics,
    plot_calibration_curves,
    plot_error_distribution,
    plot_performance_vs_quality,
    plot_confusion_matrix_pair,
)
from src.visualization.eda import (
    print_classification_summary,
    load_and_show_error_samples,
)

# --- Configuration ---
set_plot_style()
logger = get_logger(__name__)

logger.info(f"Project Root: {PROJECT_ROOT}")

In [ ]:
# 2. Hardware Verification
logger.info(f"PyTorch Version: {torch.__version__}")

if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    device_count = torch.cuda.device_count()
    logger.info(f"GPU available: {device_name} (x{device_count}), CUDA {torch.version.cuda}")
else:
    logger.warning("GPU not detected. Training will be extremely slow on CPU.")

In [ ]:
# 3. Model Selection & Execution
# --------------------------------------------------------------------------
MODEL_NAME = 'efficientnet_b3'  # 'efficientnet_b3' 'resnet50'
USE_QUALITY_FILTER = True
FORCE_RERUN = False
# --------------------------------------------------------------------------

logger.info(f"Selected Model: {MODEL_NAME}")
if USE_QUALITY_FILTER:
    logger.info(f"Quality Filter: ON (Threshold: {QUALITY_THRESHOLD})")
    suffix = "_filtered"
    quality_arg = f"--min-quality {QUALITY_THRESHOLD}"
else:
    logger.info("Quality Filter: OFF (Evaluating on ALL images)")
    suffix = ""
    quality_arg = "--min-quality 0.0"

HISTORY_PATH = REPORTS_DIR / f"history_{MODEL_NAME}.joblib"
PREDICTIONS_PATH = REPORTS_DIR / f"predictions_{MODEL_NAME}{suffix}.csv"

if PREDICTIONS_PATH.exists() and not FORCE_RERUN:
    logger.info(f"Found existing results: {PREDICTIONS_PATH.name}")
else:
    if FORCE_RERUN:
        logger.info("Force Rerun Enabled. Recomputing predictions...")
    else:
        logger.info("Predictions not found. Checking for weights...")
    
    weights_path = MODELS_DIR / f"{MODEL_NAME}_best.pth"
    
    if weights_path.exists():
        logger.info(f"Found trained weights at {weights_path}")
        logger.info("Running inference generation...")
        
        os.environ['PYTHONPATH'] = str(PROJECT_ROOT)
        
        !python -m src.models.train_model --model {MODEL_NAME} --evaluate --cached --num-workers 4 --batch-size 8 {quality_arg}
        
        standard_pred_path = REPORTS_DIR / f"predictions_{MODEL_NAME}.csv"
        if USE_QUALITY_FILTER and standard_pred_path.exists():
            shutil.move(str(standard_pred_path), str(PREDICTIONS_PATH))
            logger.info(f"Renamed output to {PREDICTIONS_PATH.name}")
            
    else:
        logger.info("No trained weights found. Starting FULL TRAINING...")
        os.environ['PYTHONPATH'] = str(PROJECT_ROOT)
        !python -m src.models.train_model --model {MODEL_NAME} --cached --num-workers 4 --batch-size 8 {quality_arg}

    if PREDICTIONS_PATH.exists():
        logger.info("Process finished successfully.")

In [ ]:
# 4. Load Artifacts & Plot Learning Curves

logger.info(f"Loading Artifacts for: {MODEL_NAME}...")

if HISTORY_PATH.exists():
    history = joblib.load(HISTORY_PATH)
    logger.info(f"Training History Loaded. Keys: {list(history.keys())}")
    
    best_loss_idx = np.argmin(history['val_loss'])
    best_kappa_idx = np.argmax(history.get('val_kappa', [-1]))
    
    logger.info(
        f"Best Val Loss: {history['val_loss'][best_loss_idx]:.4f} (Epoch {best_loss_idx+1}) | "
        f"Best Val Kappa: {history.get('val_kappa', [0])[best_kappa_idx]:.4f} (Epoch {best_kappa_idx+1})"
    )
    
    plot_learning_curves(
        history, 
        save_folder="evaluation", 
        title_prefix=f"{MODEL_NAME}_"
    )
    plt.show()
else:
    logger.error("History file not found.")

if PREDICTIONS_PATH.exists():
    preds_df = pd.read_csv(PREDICTIONS_PATH)
    logger.info(f"Predictions Loaded. Shape: {preds_df.shape}")
    display(preds_df.head())
else:
    logger.error("Predictions file not found.")

In [ ]:
# 5. Confusion Matrix & Detailed Metrics

if PREDICTIONS_PATH.exists():
    y_true = preds_df['true']
    y_pred = preds_df['pred']
    
    # Print scalar metrics and classification report
    print_classification_summary(y_true, y_pred)
    
    # Confusion matrices (Count + Normalized)
    plot_confusion_matrix_pair(
        y_true, y_pred,
        title_prefix=f"{MODEL_NAME}_",
        save_folder="evaluation",
    )
    plt.show()

In [ ]:
# 6. Diagnostic Panel (Confusion Matrix & ROC)

if PREDICTIONS_PATH.exists():
    logger.info(f"Loading predictions from: {PREDICTIONS_PATH}")
    results_df = pd.read_csv(PREDICTIONS_PATH)
    
    # Ensure columns match expected names
    logger.info(f"Columns: {results_df.columns.tolist()}")
    
    # Generate Diagnostics
    plot_model_diagnostics(
        results_df, 
        n_classes=5, 
        save_folder="evaluation", 
        title_prefix=f"{MODEL_NAME}_"
    )
    plt.show()
else:
    logger.warning(f"Predictions file not found at {PREDICTIONS_PATH}")

In [ ]:
# 7. Reliability & Abstention Analysis

if PREDICTIONS_PATH.exists():
    plot_confidence_abstention_panel(
        results_df, 
        confidence_threshold=0.7, 
        save_folder="evaluation",
        title_prefix=f"{MODEL_NAME}_"
    )
    plt.show()

In [ ]:
# 8. Sample Predictions Audit

if PREDICTIONS_PATH.exists() and 'image_name' in results_df.columns:
    load_and_show_error_samples(
        results_df, DATA_DIR,
        model_name=MODEL_NAME, save_folder="evaluation"
    )

## 9. Disease Screening Performance (Binary Classification)

In a clinical setting, the most critical decision is often **referral**: Should this patient see a specialist?
This corresponds to a binary classification task: **Healthy (No DR)** vs. **Diseased (Mild+Moderate+Severe+Proliferative)**.

We evaluate the model's ability to distinguish between these two groups.

In [ ]:
# 9. Disease Screening Performance (Binary Classification)

plot_binary_classification_metrics(
    results_df, 
    save_folder="evaluation", 
    title_prefix=f"{MODEL_NAME}_"
)
plt.show()

## 10. Reliability Metrics
We dive deeper into **how** predictions fail.
1. **Calibration**: Does a 90% confidence score actually mean 90% accuracy?
2. **Error Distribution**: Are errors small (e.g. Mild vs Moderate) or catastrophic (Healthy vs Proliferative)?

In [ ]:
# 10. Calibration & Error Analysis
prob_cols = [c for c in results_df.columns if c.startswith('prob_')]
prob_matrix = results_df[prob_cols].values

# Calibration Curves (Reliability Diagram)
plot_calibration_curves(
    results_df['true'].values, prob_matrix,
    save_folder="evaluation", title_prefix=f"{MODEL_NAME}_"
)
plt.show()

# Error Magnitude & Confidence Analysis
confidences = prob_matrix.max(axis=1)
plot_error_distribution(
    results_df['true'].values, results_df['pred'].values,
    confidences=confidences,
    save_folder="evaluation", title_prefix=f"{MODEL_NAME}_"
)
plt.show()

## 11. Performance vs. Image Quality
We verify if model performance degrades on lower quality images (blur, artifacts).
We use **Vessel Density** (extracted during preprocessing) as a proxy for image quality.

In [ ]:
# 11. Performance vs. Image Quality

if PREDICTIONS_PATH.exists():
    manifest_path = PROCESSED_DATA_DIR / "manifest.csv"
    if manifest_path.exists():
        manifest = pd.read_csv(manifest_path)
        
        plot_performance_vs_quality(
            results_df, manifest,
            quality_col='quality_score',
            save_folder="evaluation",
            title_prefix=f"{MODEL_NAME}_",
            min_samples=50
        )
        plt.show()
    else:
        logger.warning("Manifest not found. Cannot perform quality correlation analysis.")